# Logical CNOT with HGP + Surface-Code Ancilla via Homological Measurement

This notebook mirrors the structure of the getting-started guide, but implements a logical CNOT where:
- control and target are two logical qubits of the same HGP code patch,
- ancilla is a distance-3 rotated surface code,
- the CNOT is synthesized from two Pauli-product measurements: $ZZ$ (control-ancilla) then $XX$ (ancilla-target), each performed by `HomologicalMeasurement`.

We then compile to a Stim circuit and define feedforward observables for the logical CNOT output.

In [8]:
import numpy as np
import stim
from qec.code_constructions import HypergraphProductCode

from epic import (
    AllocCode,
    FreeCode,
    InitCode,
    PauliChar,
    PauliEigenState,
    PauliString,
    QECCompiler,
    ReadoutCode,
    RotatedSurfaceCode,
    StimLikeNoiseModel,
 )
from epic.modules.qec_gadgets.pauli_product_measurement.homological_measurement import HomologicalMeasurement
from epic.modules.stabilizers_codes.css_code import CSSCode

# Build a [[58,16,3]] HGP code from two [7,4,3] Hamming checks.
hamming_code = np.array(
    [
        [1, 0, 0, 1, 0, 1, 1],
        [0, 1, 0, 1, 1, 0, 1],
        [0, 0, 1, 0, 1, 1, 1],
    ],
    dtype=np.uint8,
)

hgp = HypergraphProductCode(hamming_code, hamming_code)
hx = hgp.x_stabilizer_matrix.toarray().astype(np.uint8)
hz = hgp.z_stabilizer_matrix.toarray().astype(np.uint8)

x_logicals, z_logicals = hgp.compute_logical_basis()
logical_qubits = [
    (
        x_logicals.getrow(i).toarray().ravel().astype(int).tolist(),
        z_logicals.getrow(i).toarray().ravel().astype(int).tolist(),
    )
    for i in range(x_logicals.shape[0])
]

# One HGP patch hosting both control and target logical qubits.
hgp_code = CSSCode.from_css_pcm(
    code_name="hgp",
    hx=hx,
    hz=hz,
    logical_qubits=logical_qubits,
)

# Distance-3 surface-code ancilla patch.
ancilla_code = RotatedSurfaceCode.from_distance(
    distance=3,
    code_name="ancilla_d3",
    system_coordinate=(1, 0),
)

print(f"hgp code    : [[{hgp_code.n}, {hgp_code.k}, {hgp_code.d}]]")
print(f"ancilla code: [[{ancilla_code.n}, {ancilla_code.k}, {ancilla_code.d}]]")
print(f"control logical (index 0): {hgp_code.logical_qubits[0].name}")
print(f"target logical  (index 1): {hgp_code.logical_qubits[1].name}")

hgp code    : [[58, 16, 3]]
ancilla code: [[9, 1, 3]]
control logical (index 0): hgp_lq_0
target logical  (index 1): hgp_lq_1


## Program (CNOT from ZZ then XX)

We define a gadget program in the same style as the getting-started notebook:
1. Allocate one HGP code patch with multiple logical qubits and bind `control` and `target` to two of them.
2. Allocate a surface-code ancilla patch.
3. Initialize the HGP patch in $|0_L\rangle$ and ancilla in $|+_L\rangle$.
4. Measure $\bar Z_c \bar Z_a$ with `HomologicalMeasurement`.
5. Measure $\bar X_a \bar X_t$ with `HomologicalMeasurement`.
6. Read out both patches in Z basis and free memory.

In [9]:
program = [
    AllocCode(
        target_code=hgp_code,
        code_varname="hgp_patch",
        logical_qubits_varnames=["control", "target"] + [f"hgp_aux_{i}" for i in range(2, hgp_code.k)],
    ),
    AllocCode(
        target_code=ancilla_code,
        code_varname="ancilla_patch",
        logical_qubits_varnames=["ancilla"],
    ),
    InitCode(targets=["hgp_patch"], initial_state=PauliEigenState.Z_plus, tag="init_hgp"),
    InitCode(targets=["ancilla_patch"], initial_state=PauliEigenState.X_plus, tag="init_ancilla"),
    HomologicalMeasurement(
        targets=["control", "ancilla"],
        product_to_measure=PauliString(string=(PauliChar.Z, PauliChar.Z)),
        tag="MZZ_ca",
    ),
    HomologicalMeasurement(
        targets=["ancilla", "target"],
        product_to_measure=PauliString(string=(PauliChar.X, PauliChar.X)),
        tag="MXX_at",
    ),
    ReadoutCode(targets=["hgp_patch"], tag="LZ_hgp"),
    ReadoutCode(targets=["ancilla_patch"], tag="LZ_ancilla"),
    FreeCode(code_varname="hgp_patch"),
    FreeCode(code_varname="ancilla_patch"),
]

print(f"Program length: {len(program)} gadgets")

Program length: 10 gadgets


## Compiler Configuration

As in the other homological-measurement examples, we use ZX-coloring extraction for syndrome primitives and set objective distance to 3.

In [10]:
config = {
    "objective_distance": 3,
    "primitives": {
        "epic.core.qec_primitives.interfaces.ApplyGate": "epic.modules.qec_primitives.apply_gates.simple_gate_application.SimpleGateApplication",
        "epic.core.qec_primitives.interfaces.Readout": "epic.modules.qec_primitives.readouts.naive_readout.NaiveReadout",
        "epic.core.qec_primitives.interfaces.ExtractSyndrome": "epic.modules.qec_primitives.syndrome_extraction.zxcoloring_extraction.ZXColoringExtraction",
    },
}

compiler = QECCompiler(config=config)

## Compile and Build Stim Circuit

We compile the program, inspect emitted observables, then define deterministic Stim observables corresponding to the logical CNOT action.

Feedforward convention (same idea as in getting-started):
- output control observable: direct Z readout of HGP logical qubit 0 (`control`)
- output target observable: $m(ZZ_{ca}) \oplus m(Z_a) \oplus m(Z_{\mathrm{target}})$, where target is HGP logical qubit 1

In [11]:
compiled_program = compiler.compile(program)

print("Observables emitted by compiler:")
for obs in compiled_program.observables:
    print(f"  {obs.tag}")

noise = 1e-3
noise_model = StimLikeNoiseModel.from_stim_like_probabilities(
    after_clifford_depolarization=noise,
    after_reset_flip_probability=noise,
    before_measure_flip_probability=noise,
    before_round_data_depolarization=noise,
)

stim_observables = [
    ["readout_LZ_hgp_hgp_lq_0"],
    ["hm_PP_MZZ_ca", "readout_LZ_ancilla_ancilla_d3_lq_0", "readout_LZ_hgp_hgp_lq_1"],
]

stim_program = compiled_program.to_stim_program(stim_observables, noise_model=noise_model)
circuit = stim.Circuit(stim_program)

print()
print(f"Circuit qubits      : {circuit.num_qubits}")
print(f"Circuit detectors   : {circuit.num_detectors}")
print(f"Circuit observables : {circuit.num_observables}")
print(
    "Shortest graphlike error weight:",
    len(circuit.shortest_graphlike_error(ignore_ungraphlike_errors=True)),
)

Observables emitted by compiler:
  readout_LZ_hgp_hgp_lq_3
  readout_LZ_hgp_hgp_lq_4
  readout_LZ_hgp_hgp_lq_15
  readout_LZ_hgp_hgp_lq_11
  readout_LZ_hgp_hgp_lq_5
  readout_LZ_hgp_hgp_lq_12
  readout_LZ_hgp_hgp_lq_2
  readout_LZ_hgp_hgp_lq_1
  hm_PP_MXX_at
  readout_LZ_hgp_hgp_lq_14
  readout_LZ_hgp_hgp_lq_6
  readout_LZ_hgp_hgp_lq_0
  readout_LZ_hgp_hgp_lq_8
  hm_PP_MZZ_ca
  readout_LZ_hgp_hgp_lq_10
  readout_LZ_hgp_hgp_lq_7
  readout_LZ_hgp_hgp_lq_13
  readout_LZ_hgp_hgp_lq_9
  readout_LZ_ancilla_ancilla_d3_lq_0

Circuit qubits      : 197
Circuit detectors   : 856
Circuit observables : 2
Shortest graphlike error weight: 2
